In [1]:
import sys
sys.path.append('../libs')

In [2]:
import ios

In [3]:
version = 3
PATH_NAMES = f'../../data/v{version}/a-z_zip/'
PATH_CNKI_NAMES = f'../../results/v{version}/names/chinese/CNKI_uniqueChineseGivenNames_CNKIpop.csv'
PATH_UNIQUE_CHAR_NAMES = f'../../results/v{version}/names/chinese/CNKI_uniqueChineseCharacter.csv'

# All names (Bao)

In [4]:
files = ios.listdir(PATH_NAMES, '.csv.gz')
df_names = ios.read_csv_files_from_list(files)
df_names.shape

[INFO] Found 23 files in ../../data/v3/a-z_zip/.


(65553312, 7)

In [5]:
df_names.head(2)

,id,name,pinyin,eng,NLen,NU,NG
0,99,匹,pǐ,pi,1,6.0,NaN
1,130,片,piàn,pian,1,6.0,NaN


In [6]:
npinyin = df_names.eng.nunique()
nhanzi = df_names.name.nunique()
print(f"N hanzi {nhanzi}, N pinyin {npinyin}")

N hanzi 65553312, N pinyin 160689


In [7]:
# Single chars
df_names_singlechar = df_names.query("NLen==1")
npinyin = df_names_singlechar.eng.nunique()
nhanzi = df_names_singlechar.name.nunique()
print(f"Single chars: N hanzi {nhanzi}, N pinyin {npinyin}")

Single chars: N hanzi 8096, N pinyin 401


In [8]:
# top 3
df_names_singlechar.sort_values('NU', ascending=True)[:3]

,id,name,pinyin,eng,NLen,NU,NG
7400757,1006,英,yīng,ying,1,1.5615,-0.8960
13351969,472,华,huá,hua,1,1.5777,-0.0151
7400684,190,玉,yù,yu,1,1.6045,-0.4339


# CNKI names (ours)

In [9]:
df_cnki = ios.load_csv(PATH_CNKI_NAMES)
df_cnki.shape

(39712, 4)

In [10]:
df_cnki.head(2)

,givenname,pinyin,english,CNKI_pop
0,卫民,wèimín,weimin,17
1,丽英,lìyīng#####líyīng,liying,22


In [11]:
# top 3
df_cnki.sort_values('CNKI_pop', ascending=False)[:3]

,givenname,pinyin,english,CNKI_pop
70,伟,wěi,wei,482
246,涛,tāo,tao,410
93,磊,lěi,lei,391


# Unique char names

In [12]:
df_unique_char_names = ios.load_csv(PATH_UNIQUE_CHAR_NAMES)
df_unique_char_names.shape

(8096, 18)

In [13]:
df_unique_char_names.head(2)

,character,pinyin_hanziDB,pinyin_qxk,meaning,pinyin_hanziDB_qxk,pinyin_name,pinyin_givenname,pinyin_familyname,pinyin,tone,tone_name,tone_familyname,tone_givenname,pinyin_notone,eng,eng_name,eng_familyname,eng_givenname
0,㑇,zhòu,zhòu,zhòu\t[㑳]\t1:凶狠；厉害。性情～。2:乖巧；伶俐；漂亮。,zhòu,zhòu,zhòu,zhòu,zhòu,4,4,4,4,zhou,zhou,zhou,zhou,zhou
1,㑊,yì,yì,yì\t【解（xiè）㑊】 中医学病症名。,yì,yì,yì,yì,yì,4,4,4,4,yi,yi,yi,yi,yi


In [14]:
chars = df_unique_char_names.character.unique()
wchars = df_names_singlechar.query("name in @chars").name.unique()
print(f"{len(chars)} unique single-char in our corpus vs. unique single-char in whole name db {len(wchars)}")

8096 unique single-char in our corpus vs. unique single-char in whole name db 8096


In [15]:
import unicodedata

def strip_tones(s):
    if ios.pd.isna(s):
        return s
    s = unicodedata.normalize('NFD', s)
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

has_nan = ios.pd.concat(
    [df_unique_char_names[c].map(strip_tones).str.contains(r'(?:^|#)nan(?:#|$)', na=False) for c in ['pinyin_name']],
    axis=1
).any(axis=1)

cols = ['character','pinyin_hanziDB','pinyin_name','pinyin_givenname','pinyin_familyname','eng_name','clean_eng_name']

mask = df_unique_char_names['eng_name'].isna() & has_nan
df_unique_char_names.loc[:,'clean_eng_name'] = df_unique_char_names.loc[:,'eng_name']
df_unique_char_names.loc[mask, 'clean_eng_name'] = 'nan'
print(df_unique_char_names.shape)
print(df_unique_char_names[df_unique_char_names['clean_eng_name'].isna()][cols])

df_unique_char_names = df_unique_char_names[df_unique_char_names['clean_eng_name'].notna()]
print(df_unique_char_names.shape)


(8096, 19)
     character pinyin_hanziDB pinyin_name pinyin_givenname pinyin_familyname  \
991          啵             bo         NaN              NaN               NaN   
7368         鞡             la         NaN              NaN               NaN   

     eng_name clean_eng_name  
991       NaN            NaN  
7368      NaN            NaN  
(8094, 19)


In [16]:
df_unique_char_names.sample(10)[cols]

,character,pinyin_hanziDB,pinyin_name,pinyin_givenname,pinyin_familyname,eng_name,clean_eng_name
2488,掰,bāi,bāi,bāi,bāi,bai,bai
1469,娇,jiāo,jiāo,jiāo,jiāo,jiao,jiao
6929,銮,luán,luán,luán,luán,luan,luan
3559,港,gǎng,gǎng#jiǎng,gǎng#jiǎng,gǎng#jiǎng,gang#jiang,gang#jiang
3081,楩,pián,pián,pián,pián,pian,pian
5381,脾,pí,pí,pí,pí,pi,pi
6360,谫,jiǎn,jiǎn,jiǎn,jiǎn,jian,jian
3767,炘,xīn,xìn#xīn,xìn#xīn,xìn#xīn,xin,xin
7783,鹘,gǔ,gǔ,gǔ,gǔ,gu,gu
2231,慧,huì,huì,huì,huì,hui,hui


In [17]:
df_exploded = (
    df_unique_char_names
    .assign(clean_eng_name=df_unique_char_names['clean_eng_name'].str.split('#'))
    .explode('clean_eng_name')
)
df_exploded.shape

(9044, 19)

In [18]:
# top 3
df_exploded.groupby('clean_eng_name').character.nunique().sort_values(ascending=False)[:3]

clean_eng_name
yi    146
ji    130
yu    123
Name: character, dtype: int64

In [19]:
df_exploded.query("clean_eng_name=='yi'")[cols]

,character,pinyin_hanziDB,pinyin_name,pinyin_givenname,pinyin_familyname,eng_name,clean_eng_name
1,㑊,yì,yì,yì,yì,yi,yi
29,㴔,xī,sè#yì#xī,sè#yì#xī,sè#yì#xī,se#yi#xi,yi
73,一,yī,yī,yī,yī,yi,yi
113,乂,yì,yì,yì,yì,yi,yi
117,义,yì,yì,yì,yì,yi,yi
...,...,...,...,...,...,...,...
7945,𫄷,NaN,yì,yì,yì,yi,yi
7964,𫖮,NaN,yǐ,yǐ,yǐ,yi,yi
8031,𬟁,NaN,yì,yì,yì,yi,yi
8044,𬬩,NaN,yì,yì,yì,yi,yi


In [20]:
df_exploded.clean_eng_name.nunique()

408

In [21]:
unique_single_chars = sorted(df_exploded.clean_eng_name.unique())
unique_single_chars[:5], unique_single_chars[-5:]

(['a', 'ai', 'an', 'ang', 'ao'], ['zu', 'zuan', 'zui', 'zun', 'zuo'])

In [22]:
df_exploded[cols].sort_values('clean_eng_name')

,character,pinyin_hanziDB,pinyin_name,pinyin_givenname,pinyin_familyname,eng_name,clean_eng_name
975,啊,a,ā#á#ǎ#à,ā#á#ǎ#à,ā#á#ǎ#à,a,a
801,吖,yā,yā#ā,yā#ā,yā#ā,ya#a,a
7075,锕,ā,ā,ā,ā,a,a
5387,腌,yān,ā#yān,ā#yān,ā#yān,a#yan,a
7234,阿,ā,ā#ē,ā#ē,ē,a#e,a
...,...,...,...,...,...,...,...
4659,祚,zuò,zuò,zuò,zuò,zuo,zuo
273,作,zuò,zuò,zuò,zuò,zuo,zuo
2917,柞,zhà,zuò#zhà,zuò#zhà,zuò#zhà,zuo#zha,zuo
4829,笮,zé,zuó#zé,zuó#zé,zé,zuo#ze,zuo


In [23]:
df_exploded.sort_values('pinyin_hanziDB')[cols]

,character,pinyin_hanziDB,pinyin_name,pinyin_givenname,pinyin_familyname,eng_name,clean_eng_name
975,啊,a,ā#á#ǎ#à,ā#á#ǎ#à,ā#á#ǎ#à,a,a
811,吧,ba,bā,bā,bā,ba,ba
842,呗,bei,bài,bài,bài,bai,bai
3307,汴,biàn,biàn,biàn,biàn,bian,bian
2719,昪,biàn,biàn,biàn,biàn,bian,bian
...,...,...,...,...,...,...,...
8091,𬸦,NaN,zhuó,zhuó,zhuó,zhuo,zhuo
8092,𬸪,NaN,fán,fán,fán,fan,fan
8093,𬹼,NaN,xiè,xiè,xiè,xie,xie
8094,𬺈,NaN,yǐ,yǐ,yǐ,yi,yi


In [24]:
df_exploded.query("pinyin_hanziDB.isna()")[cols]

,character,pinyin_hanziDB,pinyin_name,pinyin_givenname,pinyin_familyname,eng_name,clean_eng_name
51,䓬,NaN,zhuó,zhuó,zhuó,zhuo,zhuo
7919,𦙶,NaN,gǔ,gǔ,gǔ,gu,gu
7936,𪟝,NaN,jì,jì,jì,ji,ji
7937,𪣻,NaN,lǒu,lǒu,lǒu,lou,lou
7938,𪤗,NaN,liào,liào,liào,liao,liao
...,...,...,...,...,...,...,...
8091,𬸦,NaN,zhuó,zhuó,zhuó,zhuo,zhuo
8092,𬸪,NaN,fán,fán,fán,fan,fan
8093,𬹼,NaN,xiè,xiè,xiè,xie,xie
8094,𬺈,NaN,yǐ,yǐ,yǐ,yi,yi
